In [9]:
import logging
import numpy as np
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from google.colab import files


In [ ]:
# ===========FAISS = Facebook AI Similarity Search======================
# It is used for: Vector Search
# Instead of keyword search, it finds semantic similarity.

# BM25 (Best Matching 25) is a keyword search algorithm.

# CrossEncoder - Used for re-ranking results.

In [10]:
# 2️⃣ Upload TXT file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

print("File loaded successfully")


Saving rag_pdf.txt to rag_pdf.txt
File loaded successfully


In [11]:
# 3️⃣ Split text into chunks
raw_docs = text.split("\n\n")  # Splits document by paragraphs.

docs = [
    Document(page_content=chunk.strip())
    for chunk in raw_docs
    if chunk.strip() != ""
]

print("Number of document chunks:", len(docs))

Number of document chunks: 1


In [12]:
# 4️⃣ Create embeddings model
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
# 5️⃣ Create FAISS vector store
vectorstore = FAISS.from_documents(docs, embeddings_model)

print("Vector store created")

Vector store created


In [16]:
# 6️⃣ BM25 setup
tokenized_docs = [doc.page_content.lower().split() for doc in docs] # Tokenizer
bm25 = BM25Okapi(tokenized_docs)

print("BM25 initialized")


BM25 initialized


In [17]:
# 7️⃣ CrossEncoder setup (for re-ranking)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")

print("CrossEncoder loaded")

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CrossEncoder loaded


In [ ]:
# 8️⃣ Groq LLM setup
llm = ChatGroq(
    api_key="API_KEY",
    model="llama-3.1-8b-instant",  # Model used to generate answers.
    temperature=0
)

print("Groq LLM ready")

Groq LLM ready


In [19]:
# 9️⃣ Advanced RAG function
def advanced_rag_txt(query, top_k=3, hybrid_weight=0.5):

    # Semantic search (vector similarity)
    # Finds documents semantically similar to query.
    semantic_docs = vectorstore.similarity_search(query, k=top_k)

    # BM25 keyword search
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Hybrid scoring
    # vector search + keyword search
    hybrid_docs = {}

    for i, doc in enumerate(docs):

        semantic_score = 1 if doc in semantic_docs else 0
        bm25_score = bm25_scores[i]

        hybrid_docs[doc.page_content] = (
            hybrid_weight * semantic_score
            + (1 - hybrid_weight) * bm25_score
        )

    # CrossEncoder re-ranking
    candidate_texts = list(hybrid_docs.keys())

    cross_scores = cross_encoder.predict(
        [(query, text) for text in candidate_texts]
    )

    reranked_texts = [
        text for _, text in
        sorted(zip(cross_scores, candidate_texts), reverse=True)
    ]

    # Build context
    top_context = "\n".join(reranked_texts[:top_k])

    prompt = f"""
You are an AI assistant.
Answer the question using ONLY the provided context.

Context:
{top_context}

Question:
{query}

Answer:
"""

    # Call Groq LLM
    response = llm.invoke(prompt)

    return response.content

In [20]:
# 🔟 Ask a question
query = "Summarize the main points of the document and highlight important metrics."

answer = advanced_rag_txt(query)

print("\nFinal Answer:\n")
print(answer)


Final Answer:

**Summary of the Document:**

The document discusses two popular programming languages: Python and Java. Here are the main points:

**Python:**

1. Created by Guido van Rossum in 1991
2. Emphasizes code readability and simplicity
3. Supports multiple programming paradigms (procedural, object-oriented, and functional)
4. Dynamically typed and features automatic memory management
5. Widely used in web development, data science, artificial intelligence, and machine learning
6. Has a large standard library, strong community support, and cross-platform compatibility

**Java:**

1. Developed by Sun Microsystems and later acquired by Oracle Corporation
2. Designed around core Object-Oriented Programming (OOP) principles (encapsulation, inheritance, polymorphism, and abstraction)
3. Defines two main categories of data types: primitive and non-primitive
4. Widely used for enterprise applications, mobile development, and web systems

**Important Metrics:**

None are explicitly me